# ETL Pipeline

## Securely Mounting ADLS Gen2 Storage by OAuth
- To Access Raw Data Ingested using ADF  

In [0]:
from pyspark.sql.functions import *

In [0]:


container = "raw"
storage_account = "etlprojectspotify"
mount_name = "raw"

result=dbutils.notebook.run("/Workspace/Users/suman.kr.ghorai@gmail.com/Spotify-ETL-Databricks/utils/mount_utils", 60, {
    "container": container,
    "storage_account": storage_account,
    "mount_name": mount_name
})
print(result)

## Bronze Layer  
- Store Raw Data

In [0]:
result_bronze=dbutils.notebook.run("/Workspace/Users/suman.kr.ghorai@gmail.com/Spotify-ETL-Databricks/medallion_layers/bronze_layer", 200,)
print(result_bronze)

### Silver Layer 
- Validating Data
- Transforming Data
- Enriching Data

In [0]:
try:
    result_silver = dbutils.notebook.run("/Workspace/Users/suman.kr.ghorai@gmail.com/Spotify-ETL-Databricks/medallion_layers/silver_layer", 300)
    display(result_silver)
except Exception as e:
    print(f"Error occurred: {e}")


### Gold Layer
- Star Schema generation
- Aggregate Tables
- Kpi Creation 

In [0]:
try:
    result_gold = dbutils.notebook.run("/Workspace/Users/suman.kr.ghorai@gmail.com/Spotify-ETL-Databricks/medallion_layers/gold_layer", 500)
    display(result_gold)
except Exception as e:
    print(f"Error occurred: {e}")

### Visualisation of Data from Gold Layer by Internal Databricks visualisation Tool

In [0]:
# Load Gold Layer Tables
top_10_tracks = spark.read.format("delta").load("/mnt/gold/kpi_top_10_tracks")
avg_popularity_by_artist = spark.read.format("delta").load("/mnt/gold/kpi_avg_popularity_by_artist")
explicit_counts = spark.read.format("delta").load("/mnt/gold/kpi_explicit_track_counts")
energy_distribution = spark.read.format("delta").load("/mnt/gold/kpi_energy_distribution")
fact_track_performance = spark.read.format("delta").load("/mnt/gold/fact_track_performance")
dim_date = spark.read.format("delta").load("/mnt/gold/dim_date")


In [0]:
display(top_10_tracks.orderBy("snapshot_date", "country", "daily_rank"))


Databricks visualization. Run in Databricks to view.

In [0]:
display(avg_popularity_by_artist.limit(20))


Databricks visualization. Run in Databricks to view.

In [0]:
display(explicit_counts)


Databricks visualization. Run in Databricks to view.

In [0]:
# Join to dim_date for visualization across dates
age_popularity = fact_track_performance.join(dim_date, on="snapshot_date", how="inner") \
    .groupBy("snapshot_date") \
    .agg(
        avg("track_age_days").alias("avg_track_age"),
        avg("popularity").alias("avg_popularity")
    ).orderBy("snapshot_date")

display(age_popularity)


Databricks visualization. Run in Databricks to view.

In [0]:
pop_by_country = fact_track_performance.groupBy("country") \
    .agg(avg("popularity").alias("avg_popularity"))

display(pop_by_country)


Databricks visualization. Run in Databricks to view.

In [0]:
valence_trend = fact_track_performance.groupBy("snapshot_date", "valence_category") \
    .count().orderBy("snapshot_date")

display(valence_trend)


Databricks visualization. Run in Databricks to view.

In [0]:
dbutils.fs.unmount("/mnt/raw")
dbutils.fs.unmount("/mnt/bronze")
dbutils.fs.unmount("/mnt/gold")
dbutils.fs.unmount("/mnt/silver")
dbutils.fs.ls("/mnt")

## More Visualisations using matplotlit,seaborn

In [0]:

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import pandas as pd
import numpy as np
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Set plot styles
plt.style.use('ggplot')
sns.set(style="whitegrid")

# Color palette inspired by Spotify's branding
spotify_colors = ["#1DB954", "#191414", "#1ED760", "#509BF5", "#F573A0", "#121212", "#535353", "#B3B3B3"]

# COMMAND ----------
# Load data from the gold layer
dim_track = spark.read.format("delta").load("/mnt/gold/dim_track")
dim_artist = spark.read.format("delta").load("/mnt/gold/dim_artist")
dim_album = spark.read.format("delta").load("/mnt/gold/dim_album")
dim_date = spark.read.format("delta").load("/mnt/gold/dim_date")
fact_track_performance = spark.read.format("delta").load("/mnt/gold/fact_track_performance")
kpi_top_10_tracks = spark.read.format("delta").load("/mnt/gold/kpi_top_10_tracks")
kpi_avg_popularity_by_artist = spark.read.format("delta").load("/mnt/gold/kpi_avg_popularity_by_artist")
kpi_explicit_track_counts = spark.read.format("delta").load("/mnt/gold/kpi_explicit_track_counts")
kpi_energy_distribution = spark.read.format("delta").load("/mnt/gold/kpi_energy_distribution")

# COMMAND ----------
# 1. Energy Distribution Visualization
# Convert to pandas for visualization
energy_dist_pdf = kpi_energy_distribution.toPandas()

plt.figure(figsize=(10, 6))
ax = sns.barplot(x='energy_category', y='count', data=energy_dist_pdf, palette=spotify_colors[:3])
plt.title('Distribution of Tracks by Energy Level', fontsize=16)
plt.xlabel('Energy Category', fontsize=12)
plt.ylabel('Number of Tracks', fontsize=12)

# Add count labels on top of bars
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
# display(plt.gcf())



In [0]:

# 2. Top Artists by Average Popularity
# Get top 15 artists
top_artists_pdf = kpi_avg_popularity_by_artist.orderBy(col("avg_popularity").desc()).limit(15).toPandas()

plt.figure(figsize=(12, 8))
ax = sns.barplot(x='avg_popularity', y='artist', data=top_artists_pdf, palette=sns.color_palette("viridis", len(top_artists_pdf)))
plt.title('Top 15 Artists by Average Popularity', fontsize=16)
plt.xlabel('Average Popularity', fontsize=12)
plt.ylabel('Artist', fontsize=12)

# Add popularity score labels
for i, v in enumerate(top_artists_pdf['avg_popularity']):
    ax.text(v + 0.5, i, f'{v:.1f}', va='center', fontsize=10)

plt.tight_layout()
# display(plt.gcf())




In [0]:
# 3. Explicit vs. Clean Tracks by Country
# Convert to pandas and prepare data
explicit_pdf = kpi_explicit_track_counts.toPandas()

# Get top 10 countries by total track count
country_totals = explicit_pdf.groupby('country')['count'].sum().reset_index().sort_values('count', ascending=False)
top_countries = country_totals.head(10)['country'].tolist()

# Filter for top countries
filtered_data = explicit_pdf[explicit_pdf['country'].isin(top_countries)]

# Pivot data for stacked bar chart
pivot_data = filtered_data.pivot(index='country', columns='is_explicit', values='count').fillna(0)
if True not in pivot_data.columns:
    pivot_data[True] = 0
if False not in pivot_data.columns:
    pivot_data[False] = 0
    
pivot_data.columns = ['Clean', 'Explicit']
pivot_data = pivot_data.sort_values(by='Explicit', ascending=False)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
pivot_data.plot(kind='bar', stacked=True, ax=ax, color=[spotify_colors[2], spotify_colors[1]])
plt.title('Explicit vs. Clean Tracks by Country (Top 10 Countries)', fontsize=16)
plt.xlabel('Country', fontsize=12)
plt.ylabel('Number of Tracks', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(title='Track Type')

# Add percentage labels
for i, country in enumerate(pivot_data.index):
    total = pivot_data.loc[country, 'Clean'] + pivot_data.loc[country, 'Explicit']
    explicit_pct = (pivot_data.loc[country, 'Explicit'] / total) * 100
    
    # Position the text in the middle of the explicit part
    explicit_y_pos = pivot_data.loc[country, 'Clean'] + (pivot_data.loc[country, 'Explicit'] / 2)
    
    if pivot_data.loc[country, 'Explicit'] > 0:
        plt.text(i, explicit_y_pos, f"{explicit_pct:.1f}%", 
                 ha='center', va='center', color='white', fontweight='bold')

plt.tight_layout()
# display(plt.gcf())



In [0]:
# 4. Audio Features Correlation Heatmap
# Getting audio features from dim_track
audio_features = dim_track.select('danceability', 'energy', 'speechiness', 
                                 'acousticness', 'instrumentalness', 
                                 'liveness', 'valence', 'tempo').toPandas()

plt.figure(figsize=(10, 8))
correlation = audio_features.corr()
mask = np.triu(correlation)
sns.heatmap(correlation, annot=True, cmap='viridis', mask=mask, 
            vmin=-1, vmax=1, center=0, square=True, linewidths=.5)
plt.title('Correlation Between Audio Features', fontsize=16)
plt.tight_layout()
# display(plt.gcf())




In [0]:
# 5. Interactive Plotly Visualization - Audio Features Distribution
# Convert to pandas for visualization
audio_features_pdf = dim_track.select('name', 'danceability', 'energy', 'valence').toPandas()

# Create a plotly scatter plot
fig = px.scatter(audio_features_pdf, x='energy', y='danceability', color='valence',
                 color_continuous_scale='viridis', opacity=0.7,
                 hover_name='name', title='Track Distribution by Energy, Danceability, and Valence')

fig.update_layout(
    width=900,
    height=600,
    coloraxis_colorbar=dict(title='Valence'),
    xaxis_title='Energy',
    yaxis_title='Danceability'
)

# Display in Databricks
import plotly.io as pio
pio.renderers.default = 'databricks'
display(fig)



In [0]:

# 6. Chart using Plotly Express - Top 10 Tracks Rankings Over Time
# Get the top 10 tracks data
top_10_data = kpi_top_10_tracks.toPandas()

# Convert date if needed
if not pd.api.types.is_datetime64_any_dtype(top_10_data['snapshot_date']):
    top_10_data['snapshot_date'] = pd.to_datetime(top_10_data['snapshot_date'])

# Filter to a specific country for clarity
country_to_analyze = top_10_data['country'].iloc[0] if not top_10_data.empty else 'US'
country_data = top_10_data[top_10_data['country'] == country_to_analyze]

# Get the most frequent top 10 tracks
top_tracks = country_data['spotify_id'].value_counts().head(10).index.tolist()
filtered_data = country_data[country_data['spotify_id'].isin(top_tracks)]

# Create the visualization
fig = px.line(filtered_data, x='snapshot_date', y='daily_rank', color='name', 
              title=f'Top 10 Tracks Rankings Over Time - {country_to_analyze}')

fig.update_layout(
    xaxis_title='Date',
    yaxis_title='Daily Rank',
    yaxis=dict(autorange='reversed'),  # Reverse the y-axis so 1 is at the top
    legend_title='Track Name',
    height=600,
    width=900
)

display(fig)



In [0]:
# 7. Mode Distribution (Major vs. Minor keys)
mode_dist = dim_track.groupBy('mode_name').count().toPandas()

# Create a pie chart
plt.figure(figsize=(10, 8))
plt.pie(mode_dist['count'], labels=mode_dist['mode_name'], autopct='%1.1f%%', 
        startangle=90, colors=[spotify_colors[0], spotify_colors[1]])
plt.title('Distribution of Tracks by Mode (Major vs. Minor)', fontsize=16)
plt.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle
plt.tight_layout()
# display(plt.gcf())